# ACE workflow visualization and reporting

Creates compact, traceable figures from the model, drugability and docking outputs. Figures describe computational prioritization only.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path('/content/drive/MyDrive/ACE_Workflow')
except ImportError:
    ROOT = Path(os.environ.get('ACE_WORKFLOW_ROOT', Path.cwd())).resolve()
    if not (ROOT / 'ace_config.py').exists(): ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ace_config.py').exists()), ROOT)
if not (ROOT / 'ace_config.py').exists(): raise FileNotFoundError('Open the Google Drive-synced ACE_Workflow folder in VS Code.')
os.environ['ACE_WORKFLOW_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ace_config import *

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw

FIG = PATHS['VIS_FIGURES']; FIG.mkdir(exist_ok=True, parents=True)
drugability_file = PATHS['DRUGABILITY'] / 'ace_drugability_all_candidates.csv'
if not drugability_file.exists(): raise FileNotFoundError('Run 3-ACE-DRUGABILITY.ipynb first.')
all_candidates = pd.read_csv(drugability_file)
docking_file = PATHS['DOCKING_REPORTS'] / 'ace_1o86_integrated_docking.csv'
docking = pd.read_csv(docking_file) if docking_file.exists() else pd.DataFrame()
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
# Screening and drugability funnel by source (nested stages: each stage requires all earlier ones).
f = all_candidates.copy()
f['stage_adme'] = f.model_pass & f.admet_descriptor_pass
f['stage_glaxo'] = f.stage_adme & f.glaxo_alert_pass
summary = f.groupby('source').agg(screened=('source', 'size'), model_consensus=('model_pass', 'sum'), plus_adme_descriptors=('stage_adme', 'sum'), plus_glaxo_alert_pass=('stage_glaxo', 'sum'), final_pass=('drugability_pass', 'sum')).reset_index()
summary.to_csv(PATHS['VIS_REPORTS'] / 'screening_funnel_by_source.csv', index=False)
long = summary.melt(id_vars='source', var_name='stage', value_name='compounds')
plt.figure(figsize=(11, 5)); sns.barplot(long, x='source', y='compounds', hue='stage'); plt.xticks(rotation=20, ha='right'); plt.title('ACE screening and drugability funnel'); plt.tight_layout(); plt.savefig(FIG / 'screening_funnel_by_source.png', dpi=240); plt.show()
display(summary)

In [ ]:
# Agreement between the IC50 classifier and pChEMBL regressor.
plot_df = all_candidates.sample(min(10000, len(all_candidates)), random_state=RANDOM_STATE)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='predicted_pchembl', y='ace_active_probability', hue='source', style='drugability_pass', alpha=.55, s=22)
reg_cutoff = 9.0 - np.log10(IC50_CUTOFF_NM)  # 25 nM -> pChEMBL 7.60, same activity definition as the classifier
plt.axvline(reg_cutoff, color='black', ls='--', lw=1); plt.axhline(.5, color='black', ls='--', lw=1)
plt.title('Agreement of ACE model outputs'); plt.tight_layout(); plt.savefig(FIG / 'model_agreement.png', dpi=240); plt.show()

In [ ]:
# Docking report, when notebook 5 has completed.
if docking.empty:
    print('No docking output yet; skipping docking figures.')
else:
    top = docking.dropna(subset=['best_affinity_kcal_mol']).sort_values('best_affinity_kcal_mol').head(25).copy()
    plt.figure(figsize=(10, max(4, len(top) * .28)))
    sns.barplot(data=top, y='candidate_id', x='best_affinity_kcal_mol', hue='pose_reproducible_2A', dodge=False)
    plt.title('Top 1O86 docking affinities; colour indicates seed-pose reproducibility')
    plt.tight_layout(); plt.savefig(FIG / 'top_1o86_docking_affinities.png', dpi=240); plt.show()
    top.to_csv(PATHS['VIS_REPORTS'] / 'top_docked_candidates.csv', index=False)

In [ ]:
# 2D structures for the top final candidates (or top docked compounds when available).
if not docking.empty and docking.best_affinity_kcal_mol.notna().any():
    chosen = docking.sort_values('best_affinity_kcal_mol').head(24)
else:
    chosen = all_candidates.query('drugability_pass').sort_values('priority_score', ascending=False).head(24)
mols = [Chem.MolFromSmiles(s) for s in chosen.canonical_smiles]
labels = [str(v) for v in chosen.get('candidate_id', chosen.source_id)]
image = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(280, 220), legends=labels)
image.save(str(FIG / 'top_candidate_structures.png'))
image

In [ ]:
report = {'n_screened': int(len(all_candidates)), 'n_model_consensus': int(all_candidates.model_pass.sum()), 'n_drugability_pass': int(all_candidates.drugability_pass.sum()), 'n_docked': int(len(docking)) if not docking.empty else 0, 'figure_directory': str(FIG), 'note': 'Computational ranking is hypothesis-generating and requires experimental ACE inhibition and ADMET confirmation.'}
(PATHS['VIS_REPORTS'] / 'ace_workflow_summary.json').write_text(json.dumps(report, indent=2))
report